# 04 — Longitudinal Tracking
Quarterly risk trajectories, lab trends, and deterioration velocity analysis.

In [ ]:
import os, sys, warnings
warnings.filterwarnings("ignore")

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("pcp-panel-intelligence"):
        os.system("git clone https://github.com/Hannah-Hiltz/pcp-panel-intelligence.git")
    os.chdir("pcp-panel-intelligence")
    sys.path.insert(0, "src")
    os.system("pip install -q xgboost shap scikit-learn joblib")
    DATA_DIR  = "data/raw";      PROC_DIR  = "data/processed"
    FIG_DIR   = "reports/figures"; MODEL_DIR = "models"
    WEAR_DIR  = "data/raw/wearables"; PANEL_DIR = "data/raw/panel"
else:
    sys.path.insert(0, "../src")
    DATA_DIR  = "../data/raw";   PROC_DIR  = "../data/processed"
    FIG_DIR   = "../reports/figures"; MODEL_DIR = "../models"
    WEAR_DIR  = "../data/raw/wearables"; PANEL_DIR = "../data/raw/panel"

for d in [FIG_DIR, PROC_DIR, MODEL_DIR]:
    os.makedirs(d, exist_ok=True)

import pandas as pd, numpy as np, matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (13, 5)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
print(f"Environment: {'Colab' if IN_COLAB else 'Jupyter'}")


In [ ]:
from longitudinal_tracker import (compute_risk_trajectory, compute_lab_trends,
    flag_high_velocity_deterioration, population_trend_summary, sdoh_trajectory_analysis)

df   = pd.read_csv(f"{PANEL_DIR}/patient_panel_flat.csv")
long = pd.read_csv(f"{PROC_DIR}/longitudinal_snapshots.csv")
print(f"Longitudinal: {len(long)} rows, {long['patient_id'].nunique()} patients, {long['quarter'].nunique()} quarters")
long.head()


## Population-level risk trend Q1 → Q4

In [ ]:
pop = population_trend_summary(long)
fig, axes = plt.subplots(1,3, figsize=(14,5))

axes[0].plot(pop["quarter"], pop["avg_risk_score"], "o-", color="#378ADD", linewidth=2)
axes[0].set_title("Mean risk score by quarter", fontsize=13); axes[0].set_ylabel("Risk score")

axes[1].plot(pop["quarter"], pop["avg_hba1c"], "o-", color="#E24B4A", linewidth=2, label="HbA1c")
axes[1].plot(pop["quarter"], pop["avg_sbp"]/10, "o-", color="#EF9F27", linewidth=2, label="SBP/10")
axes[1].set_title("Lab trends Q1→Q4", fontsize=13); axes[1].legend()

axes[2].bar(pop["quarter"], pop["total_hosp"], color="#E24B4A", label="Hosp")
axes[2].bar(pop["quarter"], pop["total_er"], color="#F09595", bottom=pop["total_hosp"], label="ER")
axes[2].set_title("Utilization by quarter", fontsize=13); axes[2].legend()

plt.tight_layout()
plt.savefig(f"{FIG_DIR}/longitudinal_trends.png", dpi=150, bbox_inches="tight"); plt.show()
print(pop.to_string(index=False))


## Patient-level trajectory analysis

In [ ]:
traj = compute_risk_trajectory(long)
traj = flag_high_velocity_deterioration(traj)
print(f"Deteriorating patients:    {traj['deteriorating'].sum()} ({traj['deteriorating'].mean()*100:.1f}%)")
print(f"Improving patients:        {traj['improving'].sum()} ({traj['improving'].mean()*100:.1f}%)")
print(f"Stable patients:           {traj['stable'].sum()} ({traj['stable'].mean()*100:.1f}%)")
print(f"High-velocity deterioration: {traj['high_velocity'].sum()} ({traj['high_velocity'].mean()*100:.1f}%)")

fig, axes = plt.subplots(1,2, figsize=(13,5))
axes[0].hist(traj["risk_slope"], bins=40, color="#378ADD", edgecolor="white")
axes[0].axvline(0, color="black", linewidth=0.8, linestyle="--")
axes[0].set_title("Distribution of risk score slopes (Q1→Q4)", fontsize=13)
axes[0].set_xlabel("Risk slope (positive = deteriorating)")

axes[1].scatter(traj["q1_risk"], traj["q4_risk"],
    c=traj["risk_slope"], cmap="RdYlGn_r", alpha=0.5, s=12)
axes[1].plot([0,100],[0,100], "k--", alpha=0.4, label="No change")
axes[1].set_xlabel("Q1 risk score"); axes[1].set_ylabel("Q4 risk score")
axes[1].set_title("Q1 vs Q4 risk — above diagonal = deteriorating", fontsize=13)
axes[1].legend()

plt.tight_layout()
plt.savefig(f"{FIG_DIR}/trajectory_scatter.png", dpi=150, bbox_inches="tight"); plt.show()


## Sociology lens — deterioration rate by income quintile

In [ ]:
lab_trends = compute_lab_trends(long)
sdoh_traj  = sdoh_trajectory_analysis(long, flag_high_velocity_deterioration(compute_risk_trajectory(long)))
print("Deterioration rate by income quintile:")
print(sdoh_traj[["zip_income_quintile","n","avg_risk_slope","pct_deteriorating","avg_q4_risk","pct_high_velocity"]].to_string(index=False))

fig, axes = plt.subplots(1,2, figsize=(13,5))
axes[0].bar(sdoh_traj["zip_income_quintile"], sdoh_traj["pct_deteriorating"]*100, color="#E24B4A")
axes[0].set_title("% deteriorating by income quintile", fontsize=13)
axes[0].set_xlabel("Income quintile (1=lowest)"); axes[0].set_ylabel("%")

axes[1].bar(sdoh_traj["zip_income_quintile"], sdoh_traj["avg_risk_slope"], color="#EF9F27")
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_title("Avg risk slope by income quintile", fontsize=13)
axes[1].set_xlabel("Income quintile (1=lowest)"); axes[1].set_ylabel("Risk slope")

plt.tight_layout()
plt.savefig(f"{FIG_DIR}/sdoh_trajectory.png", dpi=150, bbox_inches="tight"); plt.show()
print("\nSociology finding: lower-income patients deteriorate faster and at higher rates,")
print("reflecting structural barriers to adherence and care access beyond clinical factors.")


## Save trajectory outputs

In [ ]:
traj.to_csv(f"{PROC_DIR}/risk_trajectories.csv", index=False)
lab_trends.to_csv(f"{PROC_DIR}/lab_trends.csv", index=False)
sdoh_traj.to_csv(f"{PROC_DIR}/sdoh_trajectory.csv", index=False)
print("Saved trajectory outputs.")
